# PROG8431: Data Analysis, Mathematics, Modeling and Algorithms
## Problem Analysis Workshop A: Statistical Testing on Autonomous Vehicle Incidents

**Team 3**
| Name | Student ID |
| :--- | :--- |
| Sultan Atanda | 9114837 |
| Ricardo Mohammed | 7500382 |
| Alamir Ibrahim | 9104960 |

<br>

**Project Topic:** Analyzing safety metrics and incident characteristics of Autonomous Vehicles (RoboTaxis) using NHTSA Standing General Order (SGO) 2021-01 data.  
**Continuous Variable:** `SV Precrash Speed (MPH)`  
**Grouping Variable:** `Crash_Interaction_Type` (Motor Vehicle vs. Fixed Object)  


# Environment Setup
These libraries must be installed in your Python kernel before you run the code in this notebook.

In [1]:
%pip install -q pandas numpy scipy plotly matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


### Import all necessary libraries and set analysis constants

In [2]:
import os
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

ALPHA = 0.05
SPEED = "SV Precrash Speed (MPH)"
GROUP = "Crash_Interaction_Type"
GROUPS = ["Motor Vehicle", "Fixed Object"]

COLORS = {"All crashes": "#64748B", "Motor Vehicle": "#2563EB", "Fixed Object": "#F97316"}
INK = "#0F172A"
TEMPLATE = "plotly_white"
FONT = dict(family="Inter, Segoe UI, Arial, sans-serif", size=15, color=INK)

# 📂 Data Loading
Loading the cleaned AV incident dataset (1,257 records, 14 features) prepared in our previous analysis. The CSV is hosted in our GitHub repo and downloaded below.

In [3]:
df = pd.read_csv('autonomous_vehicle_incidents_cleaned.csv')

group_motor = df[df['Crash_Interaction_Type'] == 'Motor Vehicle']['SV Precrash Speed (MPH)'].dropna()
group_fixed = df[df['Crash_Interaction_Type'] == 'Fixed Object']['SV Precrash Speed (MPH)'].dropna()

print(f"Motor Vehicle crashes (n={len(group_motor)})")
print(f"Fixed Object crashes (n={len(group_fixed)})")
df.head()

Motor Vehicle crashes (n=1154)
Fixed Object crashes (n=70)


,Make,Model Year,Automation System Engaged?,City,State,Roadway Type,Crash With,CP Pre-Crash Movement,SV Pre-Crash Movement,SV Precrash Speed (MPH),Within ODD?,Roadway_Conditions,Weather_Conditions,Crash_Interaction_Type
0,JAGUAR,2021,ADS,San Francisco,CA,Intersection,Passenger Car,Entering Traffic,Proceeding Straight,7,Yes,No Unusual Conditions,Clear,Motor Vehicle
1,HYUNDAI,2026,ADS,Dallas,TX,Street,SUV,Changing Lanes,Proceeding Straight,15,Yes,No Unusual Conditions,Clear,Motor Vehicle
2,HYUNDAI,2026,ADS,Dallas,TX,Street,Passenger Car,Stopped,Proceeding Straight,4,Yes,Traffic Incident,Partly Cloudy,Motor Vehicle
3,HYUNDAI,2026,ADS,Dallas,TX,Intersection,SUV,Traveling Wrong Way,Proceeding Straight,9,Yes,No Unusual Conditions,Clear,Motor Vehicle
4,HYUNDAI,2025,ADS,Austin,TX,Intersection,SUV,Entering Traffic,Proceeding Straight,21,Yes,No Unusual Conditions,Clear,Motor Vehicle


# Analysis class
`SpeedAnalysis` holds the two groups and runs each step as a method, logging every test for the final results table.

In [4]:
# @title
class SpeedAnalysis:
    """Compares pre-crash speed between two crash types: histogram, normality, F-test and t-test."""

    def __init__(self, df, value_col=SPEED, group_col=GROUP, groups=GROUPS, alpha=ALPHA):
        self.value_col, self.group_col, self.groups, self.alpha = value_col, group_col, groups, alpha
        self.all_speeds = df[value_col].dropna()
        self.data = df.loc[df[group_col].isin(groups), [group_col, value_col]].dropna()
        self.samples = {g: self.data.loc[self.data[group_col] == g, value_col] for g in groups}
        self.results = []
        self.equal_var = True

    # ---------- helpers ----------
    @staticmethod
    def fmt_p(p):
        return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

    def decide(self, test, h0, statistic, p):
        reject = p <= self.alpha
        self.results.append({"Test": test, "H₀": h0, "Statistic": statistic, "p-value": self.fmt_p(p),
                             "Decision": "Reject H₀" if reject else "Fail to reject H₀"})
        return reject

    @staticmethod
    def style_fig(fig, title, height=500):
        fig.update_layout(template=TEMPLATE, font=FONT, height=height,
                          title=dict(text=f"<b>{title}</b>", x=0.01, font=dict(size=20)),
                          legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, title=None),
                          margin=dict(l=70, r=30, t=100, b=60))
        return fig

    @staticmethod
    def verdict(text, reject):
        accent = "#DC2626" if reject else "#16A34A"
        display(HTML(f'<div style="border-left:8px solid {accent};background:#F8FAFC;color:{INK};'
                     f'padding:12px 18px;margin:10px 0;font-size:17px;border-radius:6px">{text}</div>'))

    @staticmethod
    def styled_table(table, caption=None):
        styler = table.style.set_table_styles([
            {"selector": "caption", "props": f"font-size:16px;font-weight:600;color:{INK};padding-bottom:8px"},
            {"selector": "th", "props": f"background:{INK};color:white;padding:8px 14px;font-size:14px"},
            {"selector": "td", "props": f"background:white;color:{INK};padding:8px 14px;font-size:15px;text-align:center"},
        ])
        return styler.set_caption(caption) if caption else styler

    # ---------- summary ----------
    def summary(self):
        table = (self.data.groupby(self.group_col)[self.value_col]
                     .agg(n="count", mean="mean", median="median", std="std", min="min", max="max",
                          pct_zero=lambda s: (s == 0).mean() * 100)
                     .reindex(self.groups)
                     .rename(columns={"pct_zero": "% at 0 mph"}))
        display(self.styled_table(table, "Pre-crash speed (mph)")
                .format("{:.1f}").format("{:,.0f}", subset=["n"]))

    # ---------- ① histogram ----------
    def plot_histogram(self):
        fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                            subplot_titles=["<b>All crashes</b>", "<b>By crash type</b>"])

        overall_mean = self.all_speeds.mean()
        fig.add_trace(go.Histogram(x=self.all_speeds, nbinsx=30, name="All crashes", showlegend=False,
                                   marker=dict(color=COLORS["All crashes"], line=dict(color="white", width=1))),
                      row=1, col=1)
        fig.add_vline(x=overall_mean, line=dict(color="#DC2626", width=3, dash="dash"),
                      annotation_text=f"<b>Mean: {overall_mean:.1f} mph</b>", annotation_position="top right",
                      annotation_font=dict(color="#DC2626", size=14), row=1, col=1)

        lower = min(self.samples, key=lambda g: self.samples[g].mean())
        for name, sample in self.samples.items():
            fig.add_trace(go.Histogram(x=sample, nbinsx=30, histnorm="percent", name=name, opacity=0.6,
                                       marker_color=COLORS[name]), row=1, col=2)
            fig.add_vline(x=sample.mean(), line=dict(color=COLORS[name], width=3, dash="dash"),
                          annotation_text=f"<b>{sample.mean():.1f} mph</b>",
                          annotation_position="top left" if name == lower else "top right",
                          annotation_font=dict(color=COLORS[name], size=14), row=1, col=2)

        fig.update_layout(barmode="overlay")
        fig.update_xaxes(title_text="Pre-crash speed (mph)")
        fig.update_yaxes(title_text="Number of crashes", row=1, col=1)
        fig.update_yaxes(title_text="% of crashes in group", row=1, col=2)
        self.style_fig(fig, "Distribution of Subject Vehicle pre-crash speed", height=520).show()

    # ---------- ② normality ----------
    def normality_test(self):
        panels = {"All crashes": self.all_speeds, **self.samples}
        fig = make_subplots(rows=1, cols=3, subplot_titles=[f"<b>{g}</b>" for g in panels],
                            horizontal_spacing=0.07)
        lines, any_reject = [], False
        for col, (name, sample) in enumerate(panels.items(), start=1):
            (theoretical, observed), (slope, intercept, _) = stats.probplot(sample, dist="norm")
            ends = theoretical[[0, -1]]
            fig.add_trace(go.Scatter(x=theoretical, y=observed, mode="markers", name=name,
                                     marker=dict(color=COLORS[name], size=7, opacity=0.6)), row=1, col=col)
            fig.add_trace(go.Scatter(x=ends, y=slope * ends + intercept, mode="lines", name="If perfectly normal",
                                     showlegend=(col == 1), line=dict(color=INK, width=2, dash="dash")),
                          row=1, col=col)

            w_stat, p = stats.shapiro(sample)
            reject = self.decide(f"Shapiro-Wilk ({name})", "Speed is normal", f"W = {w_stat:.3f}", p)
            any_reject |= reject
            lines.append(f"<b>{name}</b>: W = {w_stat:.3f}, {self.fmt_p(p)} → "
                         f"{'<b>reject H₀: not normal</b>' if reject else 'fail to reject H₀: consistent with normal'}")

        fig.update_xaxes(title_text="Theoretical quantiles")
        fig.update_yaxes(title_text="Observed speed (mph)", row=1, col=1)
        self.style_fig(fig, "Q-Q plots: points bending away from the dashed line = not normal").show()
        self.verdict("<br>".join(lines), any_reject)

    # ---------- ③ F-test ----------
    def f_test(self):
        (name_a, a), (name_b, b) = self.samples.items()
        var_a, var_b = a.var(ddof=1), b.var(ddof=1)
        if var_a >= var_b:
            f_ratio, df1, df2 = var_a / var_b, len(a) - 1, len(b) - 1
        else:
            f_ratio, df1, df2 = var_b / var_a, len(b) - 1, len(a) - 1
        p = min(1.0, 2 * stats.f.sf(f_ratio, df1, df2))   # two-tailed: H₁ is "variances differ"
        reject = self.decide("F-test", "Equal variances", f"F = {f_ratio:.2f}", p)
        self.equal_var = not reject

        fig = px.box(self.data, x=self.group_col, y=self.value_col, color=self.group_col, points="outliers",
                     category_orders={self.group_col: self.groups}, color_discrete_map=COLORS,
                     labels={self.value_col: "Pre-crash speed (mph)", self.group_col: ""})
        fig.update_layout(showlegend=False)
        self.style_fig(fig, f"Spread of speed: SD {a.std():.1f} mph ({name_a}) vs {b.std():.1f} mph ({name_b})").show()

        self.verdict(f"Variances: {name_a} {var_a:.1f} · {name_b} {var_b:.1f}<br>"
                     f"<b>F = {f_ratio:.2f}</b> (df {df1}, {df2}), {self.fmt_p(p)} → "
                     f"<b>{'reject H₀: variances differ' if reject else 'fail to reject H₀: variances similar'}</b><br>"
                     f"➜ Next step uses <b>{'Student' if self.equal_var else 'Welch'}'s t-test</b>", reject)

    # ---------- ④ t-score ----------
    def t_score(self, sample_group="Fixed Object", reference_group="Motor Vehicle"):
        sample = self.samples[sample_group]
        x = self.samples[reference_group].mean()
        x_bar, s, n = sample.mean(), sample.std(ddof=1), len(sample)
        se = s / np.sqrt(n)
        t = (x - x_bar) / se
        dof = n - 1
        p = 2 * stats.t.sf(abs(t), dof)
        t_crit = stats.t.ppf(1 - self.alpha / 2, dof)
        reject = self.decide("t-score", f"{reference_group} mean is typical of {sample_group}", f"T = {t:.2f}", p)

        x_max = max(4.5, abs(t) * 1.2)
        grid = np.linspace(-x_max, x_max, 600)
        density = stats.t.pdf(grid, dof)
        fig = go.Figure(go.Scatter(x=grid, y=density, mode="lines", line=dict(color=INK, width=2), showlegend=False))
        for tail in (grid <= -t_crit, grid >= t_crit):
            fig.add_trace(go.Scatter(x=grid[tail], y=density[tail], fill="tozeroy", mode="none",
                                     fillcolor="rgba(220,38,38,0.35)", name=f"Unusual (|T| > {t_crit:.2f})",
                                     showlegend=bool(tail[0])))
        for k in range(-3, 4):
            fig.add_vline(x=k, line=dict(color="#CBD5E1", width=1, dash="dot"))
        fig.add_vline(x=t, line=dict(color=COLORS[reference_group], width=4))
        fig.add_annotation(x=t, y=density.max() * 0.9, showarrow=False, xanchor="left" if t < 0 else "right",
                           text=f"<b>T = {t:.2f}<br>{reference_group} mean: {x:.1f} mph</b>",
                           font=dict(color=COLORS[reference_group], size=15))
        ticks = np.arange(-int(x_max), int(x_max) + 1)
        fig.update_xaxes(title_text=f"t-score (and equivalent mean speed for {sample_group} crashes)",
                         tickvals=ticks, ticktext=[f"{k}<br>{x_bar + k * se:.1f} mph" for k in ticks])
        fig.update_yaxes(title_text="Probability density")
        self.style_fig(fig, f"How unusual is the {reference_group} average among {sample_group} crashes?").show()

        self.verdict(f"{sample_group}: x̄ = {x_bar:.1f} mph, s = {s:.1f}, n = {n} → s/√n = {se:.2f}<br>"
                     f"<b>T = ({x:.1f} − {x_bar:.1f}) / {se:.2f} = {t:.2f}</b>, df = {dof}, "
                     f"critical ±{t_crit:.2f}, {self.fmt_p(p)} → "
                     f"<b>{'highly unusual: reject H₀' if reject else 'typical: fail to reject H₀'}</b>", reject)

    # ---------- ⑤ t-test ----------
    def t_test(self):
        (name_a, a), (name_b, b) = self.samples.items()
        mean_a, mean_b, sd_a, sd_b, n_a, n_b = a.mean(), b.mean(), a.std(ddof=1), b.std(ddof=1), len(a), len(b)

        if self.equal_var:
            pooled_var = ((n_a - 1) * sd_a**2 + (n_b - 1) * sd_b**2) / (n_a + n_b - 2)
            t_manual = (mean_a - mean_b) / np.sqrt(pooled_var * (1 / n_a + 1 / n_b))
        else:
            t_manual = (mean_a - mean_b) / np.sqrt(sd_a**2 / n_a + sd_b**2 / n_b)

        res = stats.ttest_ind(a, b, equal_var=self.equal_var)
        t_stat, p, dof = res.statistic, res.pvalue, res.df
        t_crit = stats.t.ppf(1 - self.alpha / 2, dof)
        ci_low, ci_high = res.confidence_interval(1 - self.alpha)
        test_name = "Student's t-test" if self.equal_var else "Welch's t-test"
        reject = self.decide(test_name, "Equal means", f"t = {t_stat:.2f}", p)

        self.plot_t_score(t_stat, t_crit, dof, test_name)
        self.verdict(f"{name_a} mean {mean_a:.1f} mph (SD {sd_a:.1f}) · {name_b} mean {mean_b:.1f} mph (SD {sd_b:.1f})<br>"
                     f"<b>{test_name}: t = {t_stat:.2f}</b> (manual: {t_manual:.2f}) vs critical ±{t_crit:.2f}, "
                     f"df ≈ {dof:.0f}, {self.fmt_p(p)} → "
                     f"<b>{'reject H₀: means differ' if reject else 'fail to reject H₀: no evidence means differ'}</b><br>"
                     f"Difference {mean_a - mean_b:+.1f} mph (95% CI {ci_low:+.1f} to {ci_high:+.1f})", reject)

    def plot_t_score(self, t_stat, t_crit, dof, test_name):
        fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.1,
                            subplot_titles=["<b>Our t-score on the t-distribution</b>", "<b>Mean speed ± 95% CI</b>"])
        x_max = max(4.5, abs(t_stat) * 1.15)
        x = np.linspace(-x_max, x_max, 600)
        y = stats.t.pdf(x, dof)
        fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line=dict(color=INK, width=2), showlegend=False),
                      row=1, col=1)
        for tail in (x <= -t_crit, x >= t_crit):
            fig.add_trace(go.Scatter(x=x[tail], y=y[tail], fill="tozeroy", mode="none",
                                     fillcolor="rgba(220,38,38,0.35)", name=f"Reject H₀ (|t| > {t_crit:.2f})",
                                     showlegend=bool(tail[0])), row=1, col=1)
        fig.add_vline(x=t_stat, line=dict(color="#7C3AED", width=4), row=1, col=1)
        fig.add_annotation(x=t_stat, y=y.max() * 0.9, text=f"<b>t = {t_stat:.2f}</b>", showarrow=False,
                           xanchor="right" if t_stat > 0 else "left", font=dict(color="#7C3AED", size=16),
                           row=1, col=1)

        for name, sample in self.samples.items():
            half_width = stats.t.ppf(1 - self.alpha / 2, len(sample) - 1) * stats.sem(sample)
            fig.add_trace(go.Scatter(x=[sample.mean()], y=[name], mode="markers", showlegend=False,
                                     marker=dict(color=COLORS[name], size=16),
                                     error_x=dict(type="constant", value=half_width, thickness=3, width=10)),
                          row=1, col=2)

        fig.update_xaxes(title_text="t-score", row=1, col=1)
        fig.update_xaxes(title_text="Mean pre-crash speed (mph)", row=1, col=2)
        fig.update_yaxes(title_text="Probability density", row=1, col=1)
        self.style_fig(fig, f"{test_name}: are the mean speeds different?").show()

    # ---------- ⑥ results ----------
    def results_table(self):
        color = lambda v: "color:#DC2626;font-weight:700" if v == "Reject H₀" else "color:#16A34A;font-weight:700"
        display(self.styled_table(pd.DataFrame(self.results)).hide(axis="index").map(color, subset=["Decision"]))

## Summary statistics by crash type

In [5]:
analysis = SpeedAnalysis(df)
analysis.summary()

,n,mean,median,std,min,max,% at 0 mph
Crash_Interaction_Type,,,,,,,
Motor Vehicle,"1,154",4.9,0.0,9.2,0.0,65.0,58.7
Fixed Object,70,9.7,8.5,7.6,1.0,29.0,0.0


## ① Graphical summary: distribution of pre-crash speed

---



### 📊 Histogram: Distribution of SV Precrash Speed
* **What it shows:** The distribution of the Autonomous Vehicle's speed immediately before a crash.
* **Key Observation:** The data is heavily **right-skewed**. There is a massive spike at **0 MPH** (the AV was stopped/parked) with a long tail of higher speeds.
* **Real-world takeaway:** The vast majority of RoboTaxi crashes are low-energy, stationary incidents (e.g., getting rear-ended at a light), rather than high-speed collisions.

In [6]:
plt.figure(figsize=(10, 6))
sns.histplot(df['SV Precrash Speed (MPH)'].dropna(), bins=30, kde=True, color='skyblue')
plt.title('Distribution of Subject Vehicle Pre-Crash Speed', fontsize=14)
plt.xlabel('Pre-Crash Speed (MPH)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.axvline(df['SV Precrash Speed (MPH)'].mean(), color='red', linestyle='--', label=f'Mean: {df["SV Precrash Speed (MPH)"].mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
analysis.plot_histogram()

### 📊 Histogram insight
- **Heavily right-skewed:** a massive spike at **0 mph** (the RoboTaxi was stopped) with a long tail of higher speeds.
- **Real-world takeaway:** most RoboTaxi crashes are low-energy, stationary incidents (e.g., rear-ended at a light), not high-speed collisions.
- **By crash type:** the dashed lines mark each group's mean. The t-test checks whether that gap is real or chance.

### 📐 Statistical Hypotheses (α = 0.05)

| Test | H₀ | Hₐ |
| :--- | :--- | :--- |
| **Shapiro-Wilk** (normality) | Speed is normally distributed | Speed is *not* normally distributed |
| **F-test** (variances) | σ²<sub>Motor Vehicle</sub> = σ²<sub>Fixed Object</sub> | σ²<sub>Motor Vehicle</sub> ≠ σ²<sub>Fixed Object</sub> |
| **t-test** (means) | μ<sub>Motor Vehicle</sub> = μ<sub>Fixed Object</sub> | μ<sub>Motor Vehicle</sub> ≠ μ<sub>Fixed Object</sub> |

## ② 📈 Normality: Q-Q plot and Shapiro-Wilk test

In [ ]:
analysis.normality_test()

### Normality Assessment Interpretation
The Q-Q plots deviate sharply from the straight reference line, particularly at the tails, for all crashes and for each crash type, so `SV Precrash Speed (MPH)` is not normally distributed. The Shapiro-Wilk test confirms this with p < 0.05, so we reject H₀ of normality. This right-skew is expected in real-world AV incident data: most crashes happen while the vehicle is stationary or slow, creating a spike at 0 mph and a sparse tail of faster collisions. While this violates a strict t-test assumption, our large samples make the group means approximately normal (Central Limit Theorem), so we proceed.

## ③ 🧪 F-test: Equality of Variances

$$F = \frac{s^2_{\text{larger}}}{s^2_{\text{smaller}}}, \qquad df_1 = n_1 - 1,\; df_2 = n_2 - 1$$

If F is far from 1, the spreads differ. The result decides which t-test we use.

In [ ]:
analysis.f_test()

### 📊 F-test Summary
Comparing speed variances for Motor Vehicle (n = 1,154, SD 9.17) and Fixed Object (n = 70, SD 7.60) crashes gives **F = 1.46**, **p = 0.048**. Since p < 0.05, we **reject H₀**: the variances are significantly unequal. This justifies **Welch's t-test** (`equal_var=False`) next, since Student's t-test assumes equal variances.

## ④ 🎯 t-score: How unusual is the Motor Vehicle average speed?

$$T = \frac{X - \bar{x}}{s / \sqrt{n}}$$

**X** = Motor Vehicle mean speed, and **x̄, s, n** come from the Fixed Object sample. A large |T| means X is unusual for fixed-object crashes.

In [ ]:
analysis.t_score()

### 📊 t-score Summary
Using T = (X − x̄)/(s/√n), the Motor Vehicle average (**4.9 mph**) scores **T ≈ −5.3** against the Fixed Object sample (x̄ = 9.7 mph, s = 7.6, n = 70). That is over five standard errors below the mean, far beyond ±1.99 (p < 0.001): typical car-crash speeds are **highly unusual** for fixed-object crashes.

## ⑤ ⚖️ t-test: Equality of Means (Welch's)

$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\dfrac{s_1^2}{n_1} + \dfrac{s_2^2}{n_2}}}$$

**t-score** = how many standard errors apart the two means are. If |t| is beyond the critical value, the gap is unlikely to be chance.

In [ ]:
analysis.t_test()

### 📊 t-test Summary
Welch's **t = −5.04** falls outside the critical values **±1.99** (p < 0.001), a **highly significant** result, so we **reject H₀**. Fixed Object crashes average **9.7 mph** versus **4.9 mph** for Motor Vehicle crashes, a gap of **4.8 mph** (95% CI 2.9 to 6.7 mph), so crash type **is** linked to pre-crash speed.

## ⑤ Results summary

In [ ]:
analysis.results_table()

Test,H₀,Statistic,p-value,Decision
Shapiro-Wilk (All crashes),Speed is normal,W = 0.647,p < 0.001,Reject H₀
Shapiro-Wilk (Motor Vehicle),Speed is normal,W = 0.615,p < 0.001,Reject H₀
Shapiro-Wilk (Fixed Object),Speed is normal,W = 0.890,p < 0.001,Reject H₀
F-test,Equal variances,F = 1.46,p = 0.048,Reject H₀
t-score,Motor Vehicle mean is typical of Fixed Object,T = -5.26,p < 0.001,Reject H₀
Welch's t-test,Equal means,t = -5.04,p < 0.001,Reject H₀


> # 🎯 Takeaway
> ## **RoboTaxis hit fixed objects at about twice the speed (9.7 vs 4.9 mph) of their crashes with other vehicles, and 59% of car-on-RoboTaxi crashes happen while the RoboTaxi is stopped.**